![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 14:  Clinical Data Warehousing and Analytics-Ready Data



**Health Informatics in Python** · Part IV: Advanced Topics · Module 14 of 16

---



Transactional EHR data is optimized for *recording care*, not *analyzing it*. A
**clinical data warehouse** reshapes it into an analytics-friendly structure. This
module covers **dimensional modeling** (star schemas), **computational phenotyping**,
and building an **ML-ready feature matrix**.


## Learning objectives

By the end of this module you will be able to:

1. Contrast **transactional (OLTP)** and **analytical (OLAP)** data shapes.
2. Design a **star schema** (fact + dimension tables) for clinical encounters.
3. Build a **computational phenotype** combining diagnoses, labs, and medications.
4. Assemble an **analytics-ready feature matrix** for machine learning.
5. Explain why **cohort + phenotype + features** is the standard analytics pipeline.


## Dataset

The synthetic EHR, reshaped from its transactional tables into a **star schema**
and then into a **patient-level feature matrix**.


In [1]:
# --- Self-contained synthetic EHR generator (identical to earlier parts) ---
import numpy as np
import pandas as pd

# This function creates a synthetic electronic health record (EHR) dataset for demonstration purposes.
def make_synthetic_ehr(n_patients=200, seed=42):
    rng = np.random.default_rng(seed)

    # Generate random demographic details for n_patients
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    ages  = rng.integers(18, 90, size=n_patients)
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": rng.choice(["male","female"], size=n_patients, p=[0.49,0.51]),
        "age": ages,
        "birth_year": 2026 - ages,  # Synthetic current year assumed to be 2026
    })

    # Generate encounters (visits) for each patient
    enc_rows = []
    enc_types = ["ambulatory","emergency","inpatient","wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # Each patient gets 1-4 encounters
            day = rng.integers(0, 365*3)  # Any day in 3 years from 2023-01-01
            enc_rows.append({
                "encounter_id": f"E{len(enc_rows)+1:05d}",
                "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55,0.15,0.10,0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date()
            })
    encounters = pd.DataFrame(enc_rows)

    # Generate clinical observations (labs, vitals) for encounters
    obs_defs = [("Body height","cm",150,195),("Body weight","kg",50,110),
                ("Systolic blood pressure","mmHg",100,165),("Heart rate","/min",55,100),
                ("Hemoglobin A1c","%",4.8,9.5)]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:  # 70% chance of that observation being present
                obs_rows.append({
                    "observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"],
                    "patient_id": e["patient_id"],
                    "observation": name,
                    "value": round(float(rng.uniform(lo,hi)),1),
                    "unit": unit,
                    "date": e["date"]
                })
    observations = pd.DataFrame(obs_rows)

    # Generate medical conditions ("problems") per patient
    cond_pool = ["Essential hypertension","Type 2 diabetes mellitus","Asthma",
                 "Acute bronchitis","Major depressive disorder","Osteoarthritis",
                 "Chronic kidney disease","Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        # Each patient gets 0-3 unique random diagnoses
        for c in rng.choice(cond_pool, size=rng.integers(0,4), replace=False):
            cond_rows.append({
                "condition_id": f"C{len(cond_rows)+1:05d}",
                "patient_id": pid,
                "condition": c
            })
    conditions = pd.DataFrame(cond_rows)

    # Generate medications per patient
    med_pool = ["Lisinopril","Metformin","Albuterol","Atorvastatin","Sertraline",
                "Amoxicillin","Ibuprofen","Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        # Each patient gets 0-3 unique random medications
        for m in rng.choice(med_pool, size=rng.integers(0,4), replace=False):
            med_rows.append({
                "medication_id": f"M{len(med_rows)+1:05d}",
                "patient_id": pid,
                "medication": m
            })
    medications = pd.DataFrame(med_rows)

    # Return all tables as a dictionary
    return {
        "patients": patients,
        "encounters": encounters,
        "observations": observations,
        "conditions": conditions,
        "medications": medications
    }

# Create the synthetic EHR data
ehr = make_synthetic_ehr()

# Print the number of rows for each generated table
print("Tables:", ", ".join(f"{k} ({len(v)})" for k,v in ehr.items()))


Tables: patients (200), encounters (511), observations (1804), conditions (304), medications (276)


## 13.1 Transactional vs. analytical

The operational EHR is **normalized** for fast, consistent writes — great for care,
awkward for analysis (every question needs many joins). A warehouse **denormalizes**
into a shape optimized for reads and aggregation: the **star schema**.

| | Transactional (OLTP) | Warehouse (OLAP) |
|---|---|---|
| Optimized for | writes, consistency | reads, aggregation |
| Shape | normalized, many tables | star: fact + dimensions |
| Query | "record this visit" | "visits per condition per quarter" |


## 14.2 Designing a star schema

A **star schema** is a data warehousing modeling approach characterized by a single, central **fact table**, which stores quantitative, measurable data about business events, surrounded by multiple **dimension tables**, which provide descriptive, contextual attributes for these facts. 

In this context, the _fact table_ records information about clinical encounters—each row relates to a unique healthcare encounter (such as a doctor visit, emergency visit, or inpatient admission). This is called the "**grain**" of the fact table: here, it is set as **one row per encounter**, meaning each record reflects a single, specific patient encounter.

The surrounding _dimension tables_ (such as patient, date, or encounter type) store additional data that describes, categorizes, or provides context for each encounter (like patient demographics, calendar dates, or encounter settings), enabling flexible and efficient analytical queries.

This structure allows for efficient aggregation, summarization, and slicing/dicing of the data by various dimensions, making it especially suitable for analytical workloads and reporting.


In [2]:
import pandas as pd, numpy as np

patients = ehr["patients"].copy()
enc = ehr["encounters"].copy()
enc["date"] = pd.to_datetime(enc["date"])

# ---- dimension: patient ----
dim_patient = patients.assign(
    age_band=pd.cut(patients["age"], [17,29,39,49,59,69,120],
                    labels=["18-29","30-39","40-49","50-59","60-69","70+"]))
dim_patient = dim_patient[["patient_id","sex","age","age_band"]]

# ---- dimension: date ----
dim_date = pd.DataFrame({"date": pd.date_range(enc["date"].min(), enc["date"].max())})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.to_period("Q").astype(str)
dim_date["month"] = dim_date["date"].dt.month

# ---- dimension: encounter type ----
dim_enc_type = pd.DataFrame({"encounter_type": enc["encounter_type"].unique()})
dim_enc_type["is_acute"] = dim_enc_type["encounter_type"].isin(["emergency","inpatient"])

print("Dimensions built:")
print("  dim_patient:", dim_patient.shape, "| dim_date:", dim_date.shape,
      "| dim_enc_type:", dim_enc_type.shape)

Dimensions built:
  dim_patient: (200, 4) | dim_date: (1094, 4) | dim_enc_type: (4, 2)


### Milestone 1 - the fact table

The fact table is the central, quantitative table in a star schema, where each row represents a specific business event or transaction—in this case, a single clinical encounter. 
It contains keys linking to related dimension tables (such as patient, date, and encounter type) and stores measurable data (known as measures), such as the number of observations recorded during each encounter.

In [3]:
# Build the central fact table for the star schema: one row per encounter, with foreign keys and measures

# 1. Copy the observations table for later use
obs = ehr["observations"].copy()

# 2. Calculate the number of observations recorded in each encounter;
#    this will serve as a "measure" in the fact table and is grouped by encounter_id
obs_per_enc = obs.groupby("encounter_id").size().rename("n_observations")

# 3. Merge the encounters table with the number of observations
#    - Left join ensures every encounter is kept, even if it had no observations
#    - Fill missing (NaN) counts with 0 and cast to integer
fact_encounter = (
    enc
    .merge(obs_per_enc, on="encounter_id", how="left")
    .assign(n_observations=lambda d: d["n_observations"].fillna(0).astype(int))
)

# 4. Select fact table columns:
#    - encounter_id (primary key for the fact table)
#    - patient_id, encounter_type, date (dimension keys)
#    - n_observations (measure)
fact_encounter = fact_encounter[[
    "encounter_id", "patient_id", "encounter_type", "date", "n_observations"
]]

# Display a preview of the fact table
print("FACT_ENCOUNTER (grain = one encounter):")
print(fact_encounter.head().to_string(index=False))

# Example analytical "star" query:
#    - Join the fact table to dimension tables (date, encounter type)
#    - Group by calendar quarter and acute/non-acute status
#    - Count encounters in each category
print("\nStar query example — encounters per quarter by acuity:")
star = (
    fact_encounter
    .merge(dim_date[["date", "quarter"]], on="date")
    .merge(dim_enc_type, on="encounter_type")
    .groupby(["quarter", "is_acute"]).size().rename("n").reset_index()
)
print(star.head(8).to_string(index=False))

FACT_ENCOUNTER (grain = one encounter):
encounter_id patient_id encounter_type       date  n_observations
      E00001      P1000       wellness 2025-03-17               3
      E00002      P1000     ambulatory 2023-05-21               4
      E00003      P1000     ambulatory 2025-10-05               3
      E00004      P1000     ambulatory 2023-10-22               3
      E00005      P1001     ambulatory 2024-07-22               4

Star query example — encounters per quarter by acuity:
quarter  is_acute  n
 2023Q1     False 38
 2023Q1      True  9
 2023Q2     False 35
 2023Q2      True 10
 2023Q3     False 32
 2023Q3      True 10
 2023Q4     False 28
 2023Q4      True 11


## 14.3 Computational phenotyping

A **phenotype** is a formal, algorithmic definition of "who has condition X" in a dataset,
designed to be both reproducible and executable. Phenotypes combine data from multiple sources,
including diagnoses, laboratory results, and medication prescriptions, applying logic
to systematically identify patients that meet specific clinical criteria. For instance,
real-world phenotype definitions (such as those on PheKB) may require certain diagnoses
to be present, specific lab values to exceed thresholds, and the use of corresponding medications.
Below is an implementation of a computational phenotype for type 2 diabetes, demonstrating
how multiple domains of EHR data (diagnoses, labs, medications) are used in combinatorial logic.


In [4]:
# 1. Extract the relevant condition and medication tables from the EHR
cond = ehr["conditions"]
meds = ehr["medications"]

# 2. Identify patients with a diabetes diagnosis:
#    - For each patient, check if they have *any* diagnosis of "Type 2 diabetes mellitus"
has_dx = cond["condition"].eq("Type 2 diabetes mellitus").groupby(cond["patient_id"]).any()

# 3. Identify patients with high A1c lab values:
#    - Filter hemoglobin A1c results, group by patient, take *maximum* value
#    - A1c >= 6.5 is considered diagnostic for diabetes
a1c_max = obs[obs["observation"] == "Hemoglobin A1c"].groupby("patient_id")["value"].max()
has_lab = (a1c_max >= 6.5)

# 4. Identify patients with a prescription for Metformin:
#    - For each patient, check if they have *any* record of Metformin medication
has_med = meds["medication"].eq("Metformin").groupby(meds["patient_id"]).any()

# 5. Create a phenotype DataFrame indexed by all patients
pheno = pd.DataFrame(index=patients["patient_id"])
#    - For each patient, assign whether they meet the diagnosis, lab, or medication criteria
#      (False if there is no record)
pheno["dx"]  = has_dx.reindex(pheno.index, fill_value=False)
pheno["lab"] = has_lab.reindex(pheno.index, fill_value=False)
pheno["med"] = has_med.reindex(pheno.index, fill_value=False)

# 6. Define the phenotype logic:
#    - A patient is considered to have the diabetes phenotype if they have
#      (a) a diagnosis *OR* (b) BOTH a high A1c lab *AND* Metformin prescription
#    - This represents a classic "2 out of 3" logic used in computational phenotyping
pheno["diabetes_phenotype"] = pheno["dx"] | (pheno["lab"] & pheno["med"])

# 7. Output: show counts of diabetes phenotype cases, and contribution of each criterion
print("Diabetes phenotype (multi-criteria):")
print(pheno["diabetes_phenotype"].value_counts().rename("n"))
print("\nContribution of each criterion:")
print(pheno[["dx", "lab", "med"]].sum().rename("patients_positive"))

Diabetes phenotype (multi-criteria):
diabetes_phenotype
False    137
True      63
Name: n, dtype: int64

Contribution of each criterion:
dx      40
lab    146
med     36
Name: patients_positive, dtype: int64


### Milestone 2 - an analytics-ready feature matrix

At this stage, we've constructed an analytics-ready feature matrix: a single table where each row represents a patient and each column is a feature summarizing clinically relevant traits (e.g., demographics, lab values, clinical phenotypes). This matrix is formatted so it can be directly used as input for machine learning models or statistical analyses. Continuous variables (like lab means), binary indicators (such as the diabetes phenotype), and encoded demographic variables are all included, with one row per unique patient.

In [5]:
# Build a machine learning-ready feature matrix with one row per patient:
# 
# 1. wide_obs: Pivot the observations table so that each patient's row contains the mean value
#    for each lab/measurement type as a separate column, rounded to one decimal.
wide_obs = (
    obs.pivot_table(
        index="patient_id",
        columns="observation",
        values="value",
        aggfunc="mean"
    ).round(1)
)

# 2. n_enc: Count the number of encounters (visits) for each patient.
n_enc = fact_encounter.groupby("patient_id").size().rename("n_encounters")

# 3. n_cond: Count the number of conditions recorded for each patient.
n_cond = cond.groupby("patient_id").size().rename("n_conditions")

# 4. Start with demographic data ('sex', 'age') for each patient from the dimension table,
#    set patient_id as the index. Encode sex as a binary variable indicating female.
features = (
    dim_patient.set_index("patient_id")[["sex", "age"]]
    .assign(sex=lambda d: (d["sex"] == "female").astype(int))  # 1 for female, 0 for other
    # 5. Join with the wide-format observations, encounter counts, and condition counts.
    .join(wide_obs)
    .join(n_enc)
    .join(n_cond)
    # 6. Add a binary indicator for diabetes phenotype (from previous steps).
    .join(pheno["diabetes_phenotype"].astype(int))
)

# 7. Replace missing encounter or condition counts with 0 and ensure integer type.
features[["n_encounters", "n_conditions"]] = features[["n_encounters", "n_conditions"]].fillna(0).astype(int)

# 8. Display shape and column names of the resulting feature matrix.
print("Feature matrix shape:", features.shape)
print("Columns:", list(features.columns))
features.head()

Feature matrix shape: (200, 10)
Columns: ['sex', 'age', 'Body height', 'Body weight', 'Heart rate', 'Hemoglobin A1c', 'Systolic blood pressure', 'n_encounters', 'n_conditions', 'diabetes_phenotype']


,sex,age,Body height,Body weight,Heart rate,Hemoglobin A1c,Systolic blood pressure,n_encounters,n_conditions,diabetes_phenotype
patient_id,,,,,,,,,,
P1000,1,24,163.5,74.0,81.7,8.1,140.5,4,2,0
P1001,0,73,169.7,77.2,94.2,9.0,109.0,4,1,0
P1002,1,65,151.6,62.3,78.2,NaN,NaN,1,3,1
P1003,1,49,178.2,75.4,80.6,5.2,130.1,2,3,0
P1004,1,49,169.7,93.3,60.3,7.0,147.4,3,0,0


## 14.4 The standard analytics pipeline

The overarching structure presented in this module reflects the standard workflow
found in many clinical analytics and data science projects. This process consists
of the following key stages:

```
warehouse (star schema)  →  cohort  →  phenotype  →  feature matrix  →  model
```

- **Warehouse (star schema)**: Clinical data from multiple sources is transformed
  and organized into a unified, denormalized schema, often a star schema, to enable
  efficient and flexible querying.
- **Cohort**: From the warehouse, a specific group of patients relevant to the 
  downstream analysis is selected, often based on inclusion/exclusion criteria. 
  This step narrows the population to those for whom the study question is pertinent.
- **Phenotype**: For each patient in the cohort, executable logic is used to define
  clinical phenotypes—typically the outcome, condition, or label of interest (e.g., 
  identifying patients with diabetes), based on combinations of diagnoses, labs, or
  medication records.
- **Feature matrix**: For machine learning or statistical modeling, relevant variables
  (such as demographics, lab results, and encoded phenotypes) are assembled into a 
  structured, analytic-ready table where rows correspond to patients and columns to 
  features. This tabular format can then be directly input to models.
- **Model**: Finally, predictive or explanatory models are trained on the feature matrix 
  to answer clinical questions, support decision-making, or discover new insights.

This pipeline provides a systematic approach to leverage raw clinical data for actionable 
analytics and AI applications: starting from making data accessible, then narrowing it
and labeling it for analysis, extracting relevant features, and finally applying
computational models.


## Exercises

1. Add a **dim_condition** dimension and a condition-grain fact table, then query
   condition prevalence by patient age band.
2. Tighten the diabetes phenotype to require **two** A1c values ≥ 6.5 and report how
   the cohort size changes.
3. Add **missingness indicator** columns to the feature matrix (1 if a lab was never
   recorded) — often predictive in real EHR data.



## Key takeaways

- Warehouses **denormalize** transactional data into **star schemas** for analytics.
- **Phenotypes** are executable, multi-criteria definitions of a clinical condition.
- The reusable pipeline is **warehouse → cohort → phenotype → features → model**.



---
*Next: Module 15 - Interoperability at Scale and Bulk Data.*
